In [25]:
import os
import pandas as pd
import requests

file_name = "course_lead_scoring_2026.csv"

if not os.path.exists(file_name):
    url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
    open(file_name, "wb").write(requests.get(url).content)

df = pd.read_csv(file_name)
print(df.head())

      lead_source       industry employment_status       location  \
0  organic_search     technology          employed         europe   
1    social_media     technology          employed  south_america   
2        referral         retail          employed         europe   
3        paid_ads  manufacturing           student            NaN   
4        referral  manufacturing           student  north_america   

   annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0        88160.0                         3                  4        0.64   
1        72688.0                         3                  7        0.72   
2        44697.0                         3                  4        0.58   
3            NaN                         3                  6        0.57   
4        24062.0                         4                  5        0.62   

   converted  
0          1  
1          1  
2          1  
3          0  
4          1  


In [26]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [27]:
# qustion 1
df.industry.value_counts().sort_values(ascending=False)

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
Name: count, dtype: int64

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from IPython.display import display

df.columns = df.columns.str.lower().str.replace(' ', '_')

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
        df[col] = df[col].str.lower().str.strip().str.replace(' ', '_')
    else:
        df[col] = df[col].fillna(0.0)

if 'converted' in df.columns:
    df['converted'] = df['converted'].astype(int)

In [29]:
# Q 2
numerical = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
corr_matrix = df[numerical].corr()
display(corr_matrix)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [30]:
import sklearn
from sklearn.model_selection import train_test_split
df_full_train,df_test=train_test_split(df, test_size=0.2, random_state=1)
df_train,df_val=train_test_split(df_full_train, test_size=0.25, random_state=1)

In [31]:
len(df_train),len(df_test),len(df_full_train),len(df_val)

(3000, 1000, 4000, 1000)

In [32]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [33]:
df_full_train.dtypes,df_full_train.columns

(lead_source                  object
 industry                     object
 employment_status            object
 location                     object
 annual_income               float64
 number_of_courses_viewed      int64
 interaction_count             int64
 lead_score                  float64
 converted                     int64
 dtype: object,
 Index(['lead_source', 'industry', 'employment_status', 'location',
        'annual_income', 'number_of_courses_viewed', 'interaction_count',
        'lead_score', 'converted'],
       dtype='object'))

In [40]:
category = df_train.select_dtypes(include=['object']).columns.tolist()

In [41]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[category].apply(mutual_info_converted_score)
print(mi.sort_values(ascending=False).round(2))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


In [42]:
train_numerical = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
train_categorical = df_train.select_dtypes(include=['object']).columns.tolist()

dv = DictVectorizer(sparse=False)

train_dict = df_train[train_categorical + train_numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[train_categorical + train_numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_accuracy = accuracy_score(y_val, y_pred)
print("Original Accuracy:", round(original_accuracy, 2))

Original Accuracy: 0.65


In [43]:
features_to_test = train_categorical + train_numerical
differences = {}

for f in features_to_test:
    subset_features = [col for col in features_to_test if col != f]
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[subset_features].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val[subset_features].to_dict(orient='records'))
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    acc_sub = accuracy_score(y_val, model_sub.predict(X_val_sub))
    differences[f] = original_accuracy - acc_sub

for f, diff in sorted(differences.items(), key=lambda x: x[1]):
    print(f"{f}: {diff:.5f}")

annual_income: -0.06500
industry: 0.00000
lead_score: 0.00000
location: 0.00100
lead_source: 0.00300
employment_status: 0.00300
number_of_courses_viewed: 0.00600
interaction_count: 0.05800


In [44]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

for c_val in C_values:
    model_c = LogisticRegression(solver='liblinear', C=c_val, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    acc_c = accuracy_score(y_val, model_c.predict(X_val))
    print(f"C = {c_val} -> Accuracy: {round(acc_c, 3)}")

C = 1e-06 -> Accuracy: 0.59
C = 1e-05 -> Accuracy: 0.59
C = 0.0001 -> Accuracy: 0.59
C = 0.001 -> Accuracy: 0.648
